In [3]:
import pandas as pd
import math

In [4]:
time = 3600 * 24
U_CPU = 0.23    # CPU utilization
U_RAID = 0.19   # RAID utilization

In [5]:
#Check if multi-core cpu is used
data = pd.read_csv("transactions.csv")
transaction = len(data)

cpu_time_sum = data["CPU Time"].sum()
total_time = cpu_time_sum / 3600
print(f"Ore totali: {total_time}")
# Cores per avere una utilizzazione del 23%
cores = round(cpu_time_sum / (U_CPU * time))
print(f"Core number: {cores}")
print(data.describe())


Ore totali: 50.16190833333333
Core number: 9
       Transaction ID       CPU Time    I/O on Disk  % Read Operations
count   100000.000000  100000.000000  100000.000000      100000.000000
mean     50000.500000       1.805829       2.757550          49.988560
std      28867.657797       0.250923       0.490092          40.268379
min          1.000000       0.773100       2.000000           0.000000
25%      25000.750000       1.629900       2.000000          10.000000
50%      50000.500000       1.798800       3.000000          50.000000
75%      75000.250000       1.976300       3.000000          90.000000
max     100000.000000       2.839900       4.000000         100.000000


In [6]:
system_throughput  = transaction / time 
print(f"System Throughput : {system_throughput} tps (transaction per second)")

System Throughput : 1.1574074074074074 tps (transaction per second)


Per calcolare il **Service Demand** in ambiente multi-cpu partiamo dalla **Legge dell'Utilizzazione** per sistemi multi-core:<br>
$U_i = \frac{S_i\cdot X_i}{m}$
Dove $S_i$ è il *Tempo di servizio* della risorsa i e *m* il numero di core.<br>

Poi pasiamo per la **Legge del Flusso Forzato** che stabilisce che il throughput della singola risorsa $X_i$ è uguale al throughput dell'intero sistema $X_0$ moltiplicato per il numero medio di visite $V_i$ alla risorsa:<br>
$X_I=X_0\cdot V_i$
Sostituisco e ottengo: $U_i = \frac{S_i\cdot V_i\cdot X_0}{m}$

Definiamo per la **Service Demand Law**: $D_i = S_i\cdot V_i$, in questo modo sostituisco e ottengo: $U_i = \frac{D_i\cdot X_0}{m}$
Facio la formula inversa e ottengo:<br>
$D_i = \frac{U_i\cdot m}{X_0}$

In [7]:
D_CPU = (U_CPU * cores) / system_throughput

print(f"CPU Service Demand: {D_CPU}")

CPU Service Demand: 1.7884800000000003


Ho due strade indipendenti per stimare $D_{CPU}$. Una (la media della traccia, 1.8058 s) non dipende da m: è un dato misurato. L'altra $(\frac{U_{CPU}\cdot m}{X_0}​)$ dipende da m. Il vero m è quello che le fa combaciare → m = 9.<br>
Tratto allo stesso modo il service demand pure per i dischi RAID dove so di avere 2 dischi, quindi due centri.

Si assume che la stripe unit dell'array sia maggiore della dimensione di pagina del DBMS e che gli accessi siano allineati: ogni visita logica all'array si traduce quindi in un accesso a un singolo disco fisico.<br>
È l'ipotesi che rende sensato modellare l'array come stazione a 2 serventi indipendenti (m=2): se ogni visita impegnasse entrambi i dischi, questi non sarebbero serventi indipendenti, il modello corretto sarebbe m=1

In [8]:
disks = 2
D_RAID = (U_RAID * disks) / system_throughput
print(f"RAID Service Demand: {D_RAID}")

RAID Service Demand: 0.32832


Posso calcolare il tempo di servizio del RAID tramite la **Service Demand Law**: $D_i = S_i*V_i \Rightarrow S_i = \frac{D_i}{V_i}$

In [9]:
IO_visits = data["I/O on Disk"].mean()

S_RAID = D_RAID / IO_visits

print(f"Tempo di servizio RAID: {S_RAID}")

Tempo di servizio RAID: 0.11906221102065238


**Punto 1: Realizzare un modello del sistema e ricavare**:
- Tempo di risposta medio complessivo e per-componente
- Numero Medio di job presenti nel sistema

In [10]:
# Formula dei tempi di residenza per reti aperte 
# ATTENZIONE: assume stazione a servente SINGOLO -> sovrastima con CPU a 9 core e RAID a 2 dischi.
# Calcolo di riferimento; la risposta al punto 1 e' quella del modello JMT (celle successive).
R_CPU = D_CPU / (1 - U_CPU)
R_RAID = D_RAID / (1 - U_RAID)

R_TOT = R_CPU + R_RAID

print(f"Tempo di risposta CPU: {R_CPU}")
print(f"Tempo di risposta RAID: {R_RAID}")
print(f"Tempo di risposta TOT: {R_TOT}")

# Per calcolare il numero mmedio di clienti applico la Legge di Little che mi dice: N = X * R

N = system_throughput * R_TOT

print(f"Numero di utenti medio: {N}")


Tempo di risposta CPU: 2.322701298701299
Tempo di risposta RAID: 0.4053333333333333
Tempo di risposta TOT: 2.7280346320346327
Numero di utenti medio: 3.1574474907808248


### Analisi tramite JMT
Le utilizzazioni simulate (22.45% CPU, 18.91% RAID) riproducono quelle misurate dal performance monitor (23%, 19%): il modello è validato, poiché tali valori non sono stati imposti ma emergono dalla parametrizzazione.
Sul tempo di risposta, formula analitica e simulazione divergono del ~28%. La formula $R_i=\frac{D_i}{1−U_i}$ assume una stazione a servente singolo: una richiesta si accoda non appena il servente è occupato. Qui la CPU ha 9 core, quindi una transazione si accoda solo se tutti e 9 sono occupati contemporaneamente — evento molto raro al 23% di utilizzazione. La formula è quindi conservativa su stazioni multi-server, e JMT, che modella esplicitamente gli **m** serventi, fornisce la stima corretta.<br>
Si assume come risposta al punto 1 il risultato JMT: R = 2.12 s, N = 2.42. Verifica con la legge di Little: $N=X_0\cdot R=1.1392×2.1235=2.42 \checkmark$

In [11]:
# --- Risultati del modello di simulazione JMT (JSIMgraph) ---
# Rete aperta, 2 classi, Source -> CPU (9 serventi) -> RAID (2 serventi) -> Sink
# Service time = D_{i,r} (1 visita per stazione), distribuzione esponenziale
jmt_p1 = {"X_0 [tps]": 1.1392, "R_tot [s]": 2.1235, "N": 2.4242,
          "U_CPU": 0.2245, "U_RAID": 0.1891}

analitico_p1 = {"X_0 [tps]": system_throughput, "R_tot [s]": R_TOT, "N": N,
                "U_CPU": U_CPU, "U_RAID": U_RAID}

conf = pd.DataFrame({"Analitico (R=D/(1-U))": analitico_p1, "JMT (multi-server)": jmt_p1})
conf["Scarto %"] = ((conf.iloc[:,0] / conf.iloc[:,1] - 1) * 100).round(1)
print(conf.round(4))

           Analitico (R=D/(1-U))  JMT (multi-server)  Scarto %
X_0 [tps]                 1.1574              1.1392       1.6
R_tot [s]                 2.7280              2.1235      28.5
N                         3.1574              2.4242      30.2
U_CPU                     0.2300              0.2245       2.4
U_RAID                    0.1900              0.1891       0.5


In [12]:
# Tempo di risposta per-componente dal modello JMT (index "Response Time" per stazione)
R_CPU_jmt  = 1.7829
R_RAID_jmt = 0.3439

percomp = pd.DataFrame({
    "Analitico R=D/(1-U)": {"R_CPU [s]": R_CPU, "R_RAID [s]": R_RAID, "somma [s]": R_TOT},
    "JMT (multi-server)":  {"R_CPU [s]": R_CPU_jmt, "R_RAID [s]": R_RAID_jmt,
                            "somma [s]": (R_CPU_jmt + R_RAID_jmt) if R_CPU_jmt else None},
})
print(percomp.round(4))
print(f"\nControllo: somma dei per-componente vs System Response Time JMT ({jmt_p1['R_tot [s]']})")

            Analitico R=D/(1-U)  JMT (multi-server)
R_CPU [s]                2.3227              1.7829
R_RAID [s]               0.4053              0.3439
somma [s]                2.7280              2.1268

Controllo: somma dei per-componente vs System Response Time JMT (2.1235)


Verifica di consistenza interna: la somma dei tempi di risposta per-componente (1.7829 + 0.3439 = 2.1268 s) coincide con il System Response Time misurato indipendentemente da JMT (2.1235 s) entro lo 0.16%. <br>
Si osservi inoltre che $R_{CPU}\approx D_{CPU}$ (1.783 contro 1.789): la CPU, pur essendo il collo di bottiglia, non accoda praticamente nulla, perché con 9 core al 23% è quasi impossibile che tutti i serventi siano occupati. Il RAID, meno utilizzato ma con soli 2 serventi, accumula un'attesa relativa maggiore. È la conferma diretta del perché $R=\frac{D}{1-U}$, pensata per il servente singolo, sovrastima del 28%.

Supponendo un incremento proporzionale su tutte le classi di workload, trovare il carico massimo per-classe al quale il sistema può essere soggetto mantenendo l’utilizzazione del centro collo di bottiglia inferiore o uguale al 70%.<br>

Dopo aver parsato il csv su JMT trovo due classi di workload parametrizzate in questo modo:

In [13]:
# --- Dati da JMT Workload Analysis (k-means, 2 cluster popolati da 50k obs. ciascuno) --
p = {1: 0.50, 2: 0.50}                      # peso relativo delle classi
f_cpu = {1: 0.50029, 2: 0.49971}            # "Distribution of variable values": CPU Time
f_io = {1: 0.49957, 2: 0.50043}             # "Distribution of variable values": I/O on Disk
V_r = {1: 2.75518, 2: 2.75992}              # cluster center: I/O on Disk   - Numero di visite
read_r = {1: 0.100207, 2: 0.899564}       # cluster center: % Read Operations (100.207E-1, 899.564E-1)

# U_{i,r} = U_i * f_{i,r}   ;   X_{0,r} = X_0 * p_r   ;   D_{i,r} = U_{i,r} * m_i / X_{0,r}
X_0_r     = {r: system_throughput * p[r] for r in p}
D_CPU_r  = {r: (U_CPU  * f_cpu[r] * cores) / X_0_r[r] for r in p}
D_RAID_r = {r: (U_RAID * f_io[r]  * disks) / X_0_r[r] for r in p}

print(pd.DataFrame({"X_0_r [tps]": X_0_r, "D_CPU_r [s]": D_CPU_r,
                    "D_RAID_r [s]": D_RAID_r, "V_r": V_r, "%read_r": read_r}).T.round(4))

                   1       2
X_0_r [tps]   0.5787  0.5787
D_CPU_r [s]   1.7895  1.7874
D_RAID_r [s]  0.3280  0.3286
V_r           2.7552  2.7599
%read_r       0.1002  0.8996


Il collo di bottiglia è la risorsa con l'utilizzazione maggiore $\Rightarrow$ CPU<br>
Posso trovare il throughput massimo $X_{0,max}$ impenndo un'utilizzazione massima ($U_{max}$) del 70%, invertendo la formula: $D_i = \frac{U_i\cdot m}{X_0}$ in $X_{max} = \frac{U_{max}\cdot m}{D_{CPU}}$

In [14]:
U_MAX = 0.7

X_0_max = (U_MAX * cores) / D_CPU
alpha = X_0_max / system_throughput          # fattore di scala del carico

print(f"X_0_max = {X_0_max:.4f} tps   ({X_0_max*time:,.0f} transazioni/giorno)")
print(f"alpha  = {alpha:.4f}   (verifica: U_max/U_CPU = {U_MAX/U_CPU:.4f})\n")

for r in p:
    print(f"Classe {r}: X_0_max_r = {X_0_max*p[r]:.4f} tps -> {X_0_max*p[r]*time:,.0f} transazioni/giorno")

print(f"\nVerifica che il RAID non saturi prima:")
print(f"  U_CPU  con X_0_max = {alpha*U_CPU:.2%}  (vincolo <= 70%)")
print(f"  U_RAID con X_0_max = {alpha*U_RAID:.2%}  -> la CPU resta il collo di bottiglia")



X_0_max = 3.5225 tps   (304,348 transazioni/giorno)
alpha  = 3.0435   (verifica: U_max/U_CPU = 3.0435)

Classe 1: X_0_max_r = 1.7613 tps -> 152,174 transazioni/giorno
Classe 2: X_0_max_r = 1.7613 tps -> 152,174 transazioni/giorno

Verifica che il RAID non saturi prima:
  U_CPU  con X_0_max = 70.00%  (vincolo <= 70%)
  U_RAID con X_0_max = 57.83%  -> la CPU resta il collo di bottiglia


I due cluster hanno service demand praticamente identici su CPU e RAID (1.7895 vs 1.7874; 0.3280 vs 0.3286) e pesi 50/50. Quindi il modello multi-classe collassa su quello mono-classe per tutto ciò che riguarda le utilizzazioni.<br>
Con incremento proporzionale il mix non cambia, quindi $U_i$​ scala di $\alpha$ indipendentemente dalla decomposizione del carico in classi. Un modello multi-classe non può individuare un collo di bottiglia diverso da quello mono-classe. Le classi cambiano i tempi di risposta per classe, non le utilizzazioni aggregate.

In [15]:
U_CPU_check = sum(D_CPU_r[r] * X_0_r[r] for r in p) / cores
print(f"U_CPU ricostruita dal modello multi-classe: {U_CPU_check:.4f} (misurata: {U_CPU})")

U_CPU ricostruita dal modello multi-classe: 0.2300 (misurata: 0.23)


Si valuta ora la configurazione RAID-1 rispetto al vincolo del punto 3<br>
Si vuole ora cambiare la configurazione del sistema ed adottare un array di tipo RAID-1.
E’ possibile sostenere il throughput calcolato precedentemente con un utilizzazione del centro collo di bottiglia inferiore o uguale al 70%?
- Se sì, quanti dischi sono necessari?
- Se no, quali modifiche è necessario apportare affinché il sistema risponda ai requisiti del punto 3 ed abbia tolleranza ai guasti di almeno un disco?


I dischi non cambiano. Il tempo di servizio del singolo disco $S_{RAID}=119ms$, ricavato dalla configurazione RAID-0, resta valido: cambia il livello RAID (quante visite fisiche genera una visita logica), non la tecnologia dei dischi. Senza questa ipotesi il punto 3 non è risolvibile, perché per una configurazione che non esiste ancora non si dispone di un'utilizzazione misurata da cui ricavare i demand.<br>

Serializzazione delle scritture ai mirror: irrilevante. Che le n scritture di replica avvengano in serie o in parallelo, il lavoro totale su disco è $n\cdot S$ in entrambi i casi. Cambierebbe solo il tempo di risposta percepito dalla transazione. Poiché il punto 3 pone un vincolo sulle sole utilizzazioni, il risultato è robusto rispetto a questa scelta modellistica.

In [16]:
# In RAID-1 lettura e scrittura hanno costo diverso: separo le visite medie per transazione.
A_read  = sum(p[r] * V_r[r] * read_r[r]       for r in p)   # visite di lettura
B_write = sum(p[r] * V_r[r] * (1 - read_r[r]) for r in p)   # visite di scrittura

print(f"Visite di lettura per transazione: A = {A_read:.4f}")
print(f"Visite di scrittura per transazione: B = {B_write:.4f}")
print(f"Verifica: A + B = {A_read + B_write:.4f}  vs  V medio = {IO_visits:.4f}")

# Validazione indipendente: A calcolato direttamente dalla traccia (cattura la correlazione V-%read)
A_traccia = (data["I/O on Disk"] * data["% Read Operations"] / 100).mean()
print(f"\nA dai cluster = {A_read:.5f} | A dalla traccia = {A_traccia:.5f} "
      f"| scarto {abs(A_read-A_traccia)/A_traccia:.3%}")

Visite di lettura per transazione: A = 1.3794
Visite di scrittura per transazione: B = 1.3781
Verifica: A + B = 2.7576  vs  V medio = 2.7576

A dai cluster = 1.37941 | A dalla traccia = 1.37947 | scarto 0.005%


Modello del RAID-1 con **n** dischi. Una lettura è servita da un solo disco (i dati sono replicati ovunque, quindi il carico si distribuisce sugli n); una scrittura deve essere replicata su tutti gli n dischi:<br>
$D_r^{RAID-1} = S_{RAID} \cdot V_r \cdot \left[\rho_r +n(1-\rho_r)\right]\qquad\qquad \overline{D}^{RAID-1}=\sum_r p_r\cdot D_r^{RAID-1}\qquad\qquad U(n) = \frac{X_0\cdot \overline{D}^{RAID-1}}{n} = \underbrace{\frac{X_0\cdot S_{RAID}A}{n}}_{\text{letture}}+\underbrace{X_0\cdot S_{RAID}}_{\text{scritture}}\\A = \sum_r p_r V_r \rho_r \qquad\qquad B = \sum_r p_r V_r(1-\rho_r)$<br>

Dove $\rho_r$ è la frazione di letture della classe $r$:
- $\rho_1 = 0.100207 \Rightarrow$ Le transazioni di classe 1 fanno il **10% di letture** (90% di scritture)
- $\rho_2 = 0.899564 \Rightarrow$ Le transazioni di classe 2 fanno il **90% di letture** (10% di scritture)

Dove $p_r$ è il peso della classe cioè la propabilità che una transazione presa a caso appartenga alla classe $r$ che vediamo essere equivalente $p_1=p_2=\frac{50000}{100000} = 0.5$ (%)<br>

Nel termine delle scritture la n si semplifica: il lavoro si moltiplica per n (replica) e la capacità si moltiplica per n (serventi). Aggiungere dischi a un mirror non allevia mai il carico di scrittura, ma solo quello di lettura. Ne segue un asintoto invalicabile $U_\infin = X_0 S_{RAID} B$ e la condizione di fattibilità $U_\infin \le U_{max}$ va verificata prima di cercare n:<br>

$n\ge \frac{X_0 S_{RAID}A}{U_{max} - X_0 S_{RAID}B}$

In [17]:
def D_raid1_r(r, n):
    """Service demand su un array RAID-1 di n dischi per una transazione di classe r.
       Letture: servite da 1 solo disco. Scritture: replicate su tutti gli n dischi."""
    return S_RAID * V_r[r] * (read_r[r] + n * (1 - read_r[r]))

def U_raid1(n, X):
    """Utilizzazione della stazione RAID-1 (n serventi) al throughput di sistema X."""
    D_mean = sum(p[r] * D_raid1_r(r, n) for r in p)
    return X * D_mean / n

for n in range(2, 10):
    u = U_raid1(n, X_0_max)
    print(f"n = {n} dischi -> U_RAID1 = {u:6.2%}  {'OK' if u <= U_MAX else 'NO'}")

print(f"\nAsintoto per n -> infinito: {X_0_max * S_RAID * B_write:.2%}")
print("(le scritture si replicano su n dischi e si dividono su n serventi: n si semplifica,")
print(" quindi aggiungere dischi NON riduce mai il carico di scrittura)")

n = 2 dischi -> U_RAID1 = 86.73%  NO
n = 3 dischi -> U_RAID1 = 77.08%  NO
n = 4 dischi -> U_RAID1 = 72.26%  NO
n = 5 dischi -> U_RAID1 = 69.37%  OK
n = 6 dischi -> U_RAID1 = 67.44%  OK
n = 7 dischi -> U_RAID1 = 66.06%  OK
n = 8 dischi -> U_RAID1 = 65.03%  OK
n = 9 dischi -> U_RAID1 = 64.23%  OK

Asintoto per n -> infinito: 57.80%
(le scritture si replicano su n dischi e si dividono su n serventi: n si semplifica,
 quindi aggiungere dischi NON riduce mai il carico di scrittura)


In [18]:
denom = U_MAX - X_0_max * S_RAID * B_write
if denom <= 0:
    print("IMPOSSIBILE: il solo carico di scrittura supera gia' il 70% per qualunque n.")
else:
    n_min_exact = (X_0_max * S_RAID * A_read) / denom
    n_min = math.ceil(n_min_exact)
    print(f"n minimo esatto = {n_min_exact:.3f}  ->  n = {n_min} dischi")
    print(f"Verifica: n={n_min-1} -> {U_raid1(n_min-1, X_0_max):.2%} (>70%, NO) | "
          f"n={n_min} -> {U_raid1(n_min, X_0_max):.2%} (<=70%, OK)")

n minimo esatto = 4.742  ->  n = 5 dischi
Verifica: n=4 -> 72.26% (>70%, NO) | n=5 -> 69.37% (<=70%, OK)


In [19]:
n_disks = n_min

print(f"Configurazione: CPU {cores} core + RAID-1 con {n_disks} dischi con X_0 = {X_0_max:.4f} tps\n")
print(f"  U_CPU    = {X_0_max * D_CPU / cores:.2%}   (vincolo <= 70%)")
print(f"  U_RAID-1 = {U_raid1(n_disks, X_0_max):.2%}   (vincolo <= 70%)")
print(f"  Collo di bottiglia: {'CPU' if X_0_max*D_CPU/cores >= U_raid1(n_disks, X_0_max) else 'RAID'}")
print(f"  Tolleranza ai guasti: {n_disks - 1} dischi (richiesto >= 1)  OK\n")

print("Service demand per classe sul nuovo array:")
for r in p:
    print(f"  Classe {r} (%read={read_r[r]:.1%}): D_RAID1 = {D_raid1_r(r, n_disks):.4f} s")

Configurazione: CPU 9 core + RAID-1 con 5 dischi con X_0 = 3.5225 tps

  U_CPU    = 70.00%   (vincolo <= 70%)
  U_RAID-1 = 69.37%   (vincolo <= 70%)
  Collo di bottiglia: CPU
  Tolleranza ai guasti: 4 dischi (richiesto >= 1)  OK

Service demand per classe sul nuovo array:
  Classe 1 (%read=10.0%): D_RAID1 = 1.5087 s
  Classe 2 (%read=90.0%): D_RAID1 = 0.4606 s


In [20]:
# JMT: stesso modello, arrival rate 1.7613 tps per classe, RAID-1 con 5 serventi
# service time = 1.5087 (classe 1) e 0.4606 (classe 2)
jmt_p3 = {"X_0 [tps]": 3.5257, "R_tot [s]": 3.2064, "N": 11.2575,
          "U_CPU": 0.7063, "U_RAID": 0.7003}

print(pd.DataFrame({"Analitico": {"X_0 [tps]": X_0_max, "U_CPU": U_MAX,
                                  "U_RAID": U_raid1(n_min, X_0_max)},
                    "JMT": {k: jmt_p3[k] for k in ["X_0 [tps]", "U_CPU", "U_RAID"]}}).round(4))
print(f"\nDegrado prestazionale: R passa da {jmt_p1['R_tot [s]']:.2f}s a {jmt_p3['R_tot [s]']:.2f}s "
      f"(+{jmt_p3['R_tot [s]']/jmt_p1['R_tot [s]']-1:.0%}), N da {jmt_p1['N']:.2f} a {jmt_p3['N']:.2f}")

           Analitico     JMT
X_0 [tps]     3.5225  3.5257
U_CPU         0.7000  0.7063
U_RAID        0.6937  0.7003

Degrado prestazionale: R passa da 2.12s a 3.21s (+51%), N da 2.42 a 11.26


In [21]:
# Coerenza dei risultati JMT: ricavo D dai risultati di simulazione e lo confronto col modello
print("Verifica di coerenza (D ricavato da JMT via legge dell'utilizzazione):")
print(f"  punto 1 - D_CPU  : JMT {jmt_p1['U_CPU']*cores/jmt_p1['X_0 [tps]']:.4f}  vs modello {D_CPU:.4f}")
print(f"  punto 1 - D_RAID : JMT {jmt_p1['U_RAID']*disks/jmt_p1['X_0 [tps]']:.4f}  vs modello {D_RAID:.4f}")
print(f"  punto 3 - U_CPU atteso al throughput simulato ({jmt_p3['X_0 [tps]']}): "
      f"{jmt_p3['X_0 [tps]']*D_CPU/cores:.4f}  vs JMT {jmt_p3['U_CPU']:.4f}")
print("\n-> gli scostamenti sono imputabili al rumore di simulazione, non a errori di parametrizzazione")

print(f"\nMargine sul vincolo con n={n_min}: {U_MAX - U_raid1(n_min, X_0_max):.2%} punti percentuali")
print(f"Con n={n_min+1}: U = {U_raid1(n_min+1, X_0_max):.2%}, margine {U_MAX - U_raid1(n_min+1, X_0_max):.2%}")

Verifica di coerenza (D ricavato da JMT via legge dell'utilizzazione):
  punto 1 - D_CPU  : JMT 1.7736  vs modello 1.7885
  punto 1 - D_RAID : JMT 0.3320  vs modello 0.3283
  punto 3 - U_CPU atteso al throughput simulato (3.5257): 0.7006  vs JMT 0.7063

-> gli scostamenti sono imputabili al rumore di simulazione, non a errori di parametrizzazione

Margine sul vincolo con n=5: 0.63% punti percentuali
Con n=6: U = 67.44%, margine 2.56%


Risposta: sì, è possibile, con 5 dischi. Il ramo 3.2 non si applica. La tolleranza ai guasti richiesta ($\ge$ 1 disco) è ampiamente soddisfatta: un mirror a 5 vie ne tollera 4.<br>
La fattibilità era però tutt'altro che scontata. L'asintoto al 57.80% dimostra che una risposta negativa era possibile a priori: se la frazione di scritture avesse superato il 60.5%, nessun numero di dischi sarebbe bastato. Il workload ha il 49.97% di scritture, sotto soglia ma non di molto.
Criticità: la configurazione è al limite. Il margine sul vincolo è di soli 0.63 punti percentuali. Con 6 dischi si otterrebbe 67.44%, margine 4 volte più ampio al costo di un disco.
Costo della soluzione. Servono 5 dischi per un array che ne richiederebbe 2, con capacità utile pari a quella di un solo disco (storage 5×). Il degrado prestazionale è marcato: $R$ passa da 2.12 s a 3.21 s (+51%) e $N$ da 2.42 a 11.26 job.
Il mirroring viene pagato dal demand della classe 1 (90% scritture) è 3.28× quello della classe 2 (1.5087 s contro 0.4606 s) — divergenza invisibile nei punti 1 e 2, dove le classi erano indistinguibili. È qui che il clustering ripaga: senza la decomposizione in classi non sarebbe possibile attribuire il costo.
Osservazione critica. La radice del problema è che in RAID-1 le scritture si replicano su n dischi e si dividono su n serventi: la n si semplifica e resta un pavimento invalicabile. Un RAID-10, replicando ogni scrittura su 2 soli dischi e distribuendo le coppie via striping, eliminerebbe l'asintoto. Il testo impone RAID-1, ma su un workload write-intensive come questo la scelta è strutturalmente sfavorevole.